In [1]:
import numpy as np
import pandas as pd
from models import DecisionTree

In [2]:
class RandomForest:
    def __init__(self,n_trees,max_depth=50,min_samples=10,n_features=None,random_state=42):
        self.n_trees=n_trees
        self.max_depth=max_depth
        self.min_samples=min_samples
        self.n_features=n_features
        self.rng=np.random.RandomState(random_state)
        self.trees=[]
    def fit(self,X,y):       
        self.X=np.asarray(X)
        self.y=np.asarray(y)
        m,n=self.X.shape
        if self.n_features is not None:
            self.n_features=max(1,min(self.n_features,n))
        else:
            self.n_features=int(np.sqrt(n))
        for _ in range(self.n_trees):
            idxs=self.rng.choice(range(m),m, replace=True)
            X_sample,y_sample=self.X[idxs],self.y[idxs]
            tree=DecisionTree(max_depth=self.max_depth,min_samples=self.min_samples,n_features=self.n_features)
            tree.fit(X_sample,y_sample)
            self.trees.append(tree)
    def predict(self,X):
        X=np.asarray(X)
        preds=[]
        for x in X:
            classes,counts=np.unique([tree.predict([x])[0]for tree in self.trees],return_counts=True)
            preds.append(classes[np.argmax(counts)])
        return preds

In [3]:
df=pd.read_csv('data/iris.csv')

In [4]:
X,y=df.loc[:,['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']],df.loc[:,'Species']

In [5]:
y=y.map({'Iris-setosa':0,'Iris-versicolor':1,'Iris-virginica':2})

In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,stratify=y)

In [7]:
rf=RandomForest(n_trees=100,max_depth=10,min_samples=5)
rf.fit(X_train.values,y_train.values)
preds=rf.predict(X_test.values)
np.mean(preds==y_test.values)

np.float64(0.9333333333333333)